## SilverWorkIncremental
Incremental Silver processing using only new Bronze rows.

### Step 1 — Imports and setup
This cell imports Spark, Window, and Delta helpers, and creates a silver_run_id for the current run.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime
from delta.tables import DeltaTable
import uuid

In [0]:
silver_run_id = str(uuid.uuid4())

In [0]:
print(f"Current Silver Notebook Run ID: {silver_run_id}")

In [0]:
def get_last_processed_at(table_id: int):
    processing_control_df = (
        spark.read.table("silver.control.processing_control")
        .filter((F.col("table_id") == F.lit(table_id)) & (F.col("status") == "success"))
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    last_processed_at = processing_control_df.collect()

    if not last_processed_at:
        return None
    return last_processed_at[0]["last_processed_at"]

In [0]:
def incremental_data(table_id: int, bronze_table: str):
    last_processed_at = get_last_processed_at(table_id)
    bronze_df = spark.read.table(bronze_table)

    if not last_processed_at:
        return bronze_df, last_processed_at

    incremental_df = bronze_df.filter(
        F.col("bronze_ingested_at") > F.lit(last_processed_at)
    )
    return incremental_df, last_processed_at

In [0]:
def upsert_processing_control(
    run_id: str,
    table_id: int,
    last_processed_at: datetime,
    rows_merged: int,
    status: str,
):
    data = [
        (
            run_id,
            table_id,
            last_processed_at,
            rows_merged,
            status,
            datetime.utcnow(),
        ),
    ]

    schema = """
        run_id STRING,
        table_id BIGINT,
        last_processed_at TIMESTAMP,
        rows_merged BIGINT,
        status STRING, 
        updated_at TIMESTAMP
    """
    source_df = spark.createDataFrame(data, schema)
    target_tbl = DeltaTable.forName(spark, "silver.control.processing_control")

    target_tbl.alias("t").merge(
        source_df.alias("s"), "t.table_id = s.table_id and t.run_id = s.run_id"
    ).whenMatchedUpdate(
        set={
            "t.last_processed_at": "s.last_processed_at",
            "t.rows_merged": "s.rows_merged",
            "t.status": "s.status",
            "t.updated_at": "s.updated_at",
        }
    ).whenNotMatchedInsertAll().execute()

In [0]:
def load_data(source_df, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        DeltaTable.forName(spark, target_table).alias("t").merge(
            source_df.alias("s"), f"t.{join_key} = s.{join_key}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        source_df.write.format("delta").saveAsTable(target_table)

In [0]:
def process_cleaning(table_name, incremental_df):
    if table_name == "orders":
        cleaned_df = (
            incremental_df.withColumn(
                "order_status", F.upper(F.trim(F.col("order_status")))
            )
            .withColumn(
                "order_status",
                F.when(F.col("order_status") == F.lit(""), F.lit(None)).otherwise(
                    F.col("order_status")
                ),
            )
            .withColumn("order_amount", F.trim(F.col("order_amount")))
            .withColumn(
                "order_amount",
                F.regexp_replace(F.col("order_amount"), "[^0-9]", ""),
            )
            .withColumn(
                "order_amount",
                F.when(F.col("order_amount") == F.lit(""), F.lit(None)).otherwise(
                    F.col("order_amount").cast("double")
                ),
            )
        )
        return cleaned_df

    elif table_name == "products":
        return incremental_df
    elif table_name == "payments":
        return incremental_df
    else:
        return incremental_df

In [0]:
def process_validation(table_name, cleaned_df):
    if table_name == "orders":
        validated_df = cleaned_df.withColumn(
            "issue",
            F.when(F.col("order_status").isNull(), F.lit("order_status_null"))
            .when(F.col("order_amount").isNull(), F.lit("order_amount_null"))
            .otherwise(F.lit("No Issue")),
        )
        return validated_df

    elif table_name == "products":
        return cleaned_df.withColumn(
            "issue",
            F.lit("No Issue"),
        )
    elif table_name == "payments":
        return cleaned_df.withColumn(
            "issue",
            F.lit("No Issue"),
        )
    else:
        return cleaned_df.withColumn(
            "issue",
            F.lit("No Issue"),
        )

In [0]:
object_list = (
    spark.read.table("silver.control.object_list")
    .filter(F.col("layer") == "silver")
    .collect()
)

for row in object_list:
    table_id = row["table_id"]
    table_name = row["table_name"]
    join_key = row["join_key"]

    incremental_df, last_processed_at = incremental_data(
        table_id, f"bronze.raw.{table_name}_raw"
    )

    incremental_count = incremental_df.count()

    if incremental_count > 0:
        cleaned_df = process_cleaning(table_name, incremental_df)
        cleaned_df = (
            cleaned_df.withColumn("bronze_source_table", F.lit(table_name))
            .withColumn("silver_processed_at", F.current_timestamp())
            .withColumn("silver_run_id", F.lit(silver_run_id))
        )
        load_data(cleaned_df, f"silver.cleaned.{table_name}_cleaned", join_key)

        validated_df = process_validation(table_name, cleaned_df)
        correct_df = validated_df.filter(F.col("issue") == F.lit("No Issue"))

        load_data(correct_df, f"silver.transformed.{table_name}_transformed", join_key)

        bad_record_df = (
            validated_df.filter(F.col("issue") != F.lit("No issues"))
            .withColumn("silver_run_id", F.lit(silver_run_id))
            .withColumn("quarantine_ts", F.current_timestamp())
        )

        bad_record_df.write.mode("append").option("mergeSchema", True).saveAsTable(
            f"silver.bad_records.{table_name}_bad_records"
        )

        max_ts = incremental_df.agg(
            F.max("bronze_ingested_at").alias("max_ts")
        ).collect()[0]["max_ts"]

        upsert_processing_control(
            silver_run_id, table_id, max_ts, incremental_count, "success"
        )

    else:
        upsert_processing_control(
            silver_run_id, table_id, last_processed_at, incremental_count, "success"
        )

In [0]:
%sql
SELECT count(*) FROM bronze.raw.orders_raw

In [0]:
%sql
SELECT count(*) FROM silver.cleaned.orders_cleaned